# Homework Starter — Stage 10b: Time Series & Classification
Fill in the TODOs. Use your own project dataset or adapt the synthetic generator below.

In [ ]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install numpy
# !pip install pandas
# !pip install seaborn
# !pip install matplotlib
# !pip install scikit-learn

In [ ]:
# Imports
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split, TimeSeriesSplit
np.random.seed(7); sns.set(); plt.rcParams['figure.figsize']=(9,4)

## Option A: Use Your Project Data (Recommended)
Load your data here (ensure a DateTime index for time series).

In [ ]:
# TODO: load your data
# df = pd.read_csv('path/to.csv', parse_dates=['Date'], index_col='Date')
df = pd.read_csv(
    "../../project/data/raw/market_returns_spy_spmo.csv"
)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

df.head()

## Option B: Synthetic Generator (Use if you don't have data ready)

In [ ]:
# Synthetic series with regimes & jumps
n=500
dates=pd.bdate_range('2021-01-01', periods=n)
mu = np.where(np.arange(n)<n//2, 0.0003, -0.0001)
sigma = np.where(np.arange(n)<n//2, 0.01, 0.015)
eps = np.random.normal(mu, sigma)
jumps = np.zeros(n); jump_days = np.random.choice(np.arange(20,n-20), size=5, replace=False)
jumps[jump_days] = np.random.normal(0,0.05,size=len(jump_days))
rets = eps + jumps
price = 100*np.exp(np.cumsum(rets))
df = pd.DataFrame({'price':price}, index=dates)
df['ret'] = df['price'].pct_change().fillna(0.0)
df['log_ret'] = np.log1p(df['ret'])
df.head()

## Feature Engineering

In [ ]:
# Feature 1: previous month's momentum excess return
df["lag_1"] = df["excess_return"].shift(1)

# Feature 2: trailing 3-month average momentum performance
df["roll_mean_3"] = (
    df["excess_return"]
    .rolling(3)
    .mean()
    .shift(1)
)

# Feature 3: trailing 3-month market volatility
df["spy_vol_3"] = (
    df["spy_return"]
    .rolling(3)
    .std()
    .shift(1)
)

# Target: 1 if momentum underperforms SPY next month
df["next_excess_return"] = df["excess_return"].shift(-1)

df["y_underperform"] = (
    df["next_excess_return"] < 0
).astype(int)

df_feat = df.dropna().copy()

df_feat[
    [
        "date",
        "excess_return",
        "lag_1",
        "roll_mean_3",
        "spy_vol_3",
        "y_underperform"
    ]
].head()

## Split

In [ ]:
features = [
    "lag_1",
    "roll_mean_3",
    "spy_vol_3"
]

cut = int(len(df_feat) * 0.8)

train = df_feat.iloc[:cut]
test = df_feat.iloc[cut:]

X_train = train[features]
X_test = test[features]

y_train = train["y_underperform"]
y_test = test["y_underperform"]

print("Train:", train["date"].min(), "to", train["date"].max())
print("Test:", test["date"].min(), "to", test["date"].max())

print("Train size:", len(train))
print("Test size:", len(test))

## Pipeline + Model (Choose one track below)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

clf = Pipeline([
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression(max_iter=1000))
])

clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1 Score:  {f1:.3f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Momentum Underperformance Confusion Matrix")
plt.show()

## Interpretation (Markdown)
- What worked?
- Where might assumptions fail?
- How would you extend features or model?

### Save Notebook
Remember to save as `homework10b_modeling-time-series-and-classification_submission.ipynb`.